# TFM · Predicción de Consumo y Precio Eléctrico — Madrid
## Notebook 05 · Modelos SOTA: N-HiTS y TFT (v4)

**Correcciones respecto a v3:**
- ✅ Covariables futuras generadas directamente (no por merge)
- ✅ Compatible con GPU T4 en Colab
- ✅ `futr_df` sin nulos garantizado

**Modelos:** N-HiTS + TFT via `neuralforecast`

**Datos:** datasets procesados del notebook 03

---
## 0. Setup — Activar GPU T4 en Colab antes de ejecutar

In [ ]:
!pip install neuralforecast utilsforecast --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json, pickle, warnings, holidays
warnings.filterwarnings('ignore')

from sklearn.metrics import mean_absolute_error, mean_squared_error
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS, TFT
from neuralforecast.losses.pytorch import HuberLoss, MAE
import torch

plt.rcParams.update({
    'figure.figsize':    (14, 4),
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'font.size':         11,
})
COLORS    = ['#2563EB', '#F59E0B', '#10B981', '#EF4444', '#8B5CF6']
HORIZONTE = 48
N_EVAL    = 100

GPU = torch.cuda.is_available()
print(f'✅ Setup completado')
print(f'   GPU: {GPU} — {torch.cuda.get_device_name(0) if GPU else "CPU (recomendado GPU T4)"}')
if not GPU:
    print('   ⚠️  Sin GPU: ve a Entorno de ejecución → Cambiar tipo → GPU T4')

---
## 1. Carga de datos procesados (notebook 03)

In [ ]:
def cargar(path):
    df = pd.read_csv(path)
    df['datetime'] = pd.to_datetime(df['datetime'], utc=True).dt.tz_convert('Europe/Madrid')
    return df.set_index('datetime').sort_index()

train_scaled = cargar('data/processed/train_scaled.csv')
val_scaled   = cargar('data/processed/val_scaled.csv')
test_scaled  = cargar('data/processed/test_scaled.csv')
train_raw    = cargar('data/processed/train.csv')
val_raw      = cargar('data/processed/val.csv')
test_raw     = cargar('data/processed/test.csv')

with open('data/models/scaler_precio.pkl',  'rb') as f: scaler_precio  = pickle.load(f)
with open('data/models/scaler_demanda.pkl', 'rb') as f: scaler_demanda = pickle.load(f)
with open('data/models/metadata.json')      as f: meta = json.load(f)

df_full = pd.concat([train_scaled, val_scaled, test_scaled]).sort_index()
df_raw  = pd.concat([train_raw,    val_raw,    test_raw   ]).sort_index()

# Eliminar columnas de texto, leakage y targets
LEAKAGE = ['precio_eur_mwh', 'precio_eur_kwh', 'demanda_mw']
TARGETS = [c for c in df_full.columns if c.startswith('target_')]
TEXTO   = ['tipo_dia', 'nombre_festivo', 'anio_str']
df_full = df_full.drop(columns=[c for c in LEAKAGE+TARGETS+TEXTO if c in df_full.columns])

print(f'✅ Datos cargados: {df_full.shape[1]} features')
print(f'   Período: {df_full.index.min().date()} → {df_full.index.max().date()}')

In [ ]:
# ── Separar covariables futuras e históricas ──────────────────────────────────
# FUTURAS: deterministas, se pueden calcular para cualquier fecha futura
# (hora, día semana, mes, festivos, encoding cíclico)
FUTURE_COVS = sorted([
    c for c in df_full.columns
    if any(x in c for x in [
        'hora', 'dia_semana', 'mes', 'anio', 'semana',
        '_sin', '_cos', 'festivo', 'vispera', 'post_festivo',
        'dias_hasta', 'es_hora_solar', 'semana_anio'
    ])
])

# HISTÓRICAS: solo disponibles en el pasado
# (lags, rolling, diferencias, meteorología)
HIST_COVS = sorted([
    c for c in df_full.columns
    if c not in FUTURE_COVS
])

print(f'Covariables futuras  ({len(FUTURE_COVS)}): {FUTURE_COVS[:5]}...')
print(f'Covariables históricas ({len(HIST_COVS)}): {HIST_COVS[:5]}...')

In [ ]:
# ── Formato long para neuralforecast ─────────────────────────────────────────
def preparar_long(target_col):
    df_long = df_full.copy()
    df_long['y'] = df_raw[target_col]  # Target sin escalar
    df_long = df_long.reset_index().rename(columns={'datetime': 'ds'})
    df_long.insert(0, 'unique_id', 'Madrid')
    df_long['ds'] = df_long['ds'].dt.tz_localize(None)
    return df_long.dropna(subset=['y'])

df_precio  = preparar_long('precio_eur_mwh')
df_demanda = preparar_long('demanda_mw')

n_total = len(df_precio)
n_val   = int(n_total * 0.15)
n_test  = int(n_total * 0.15)

print(f'✅ Formato long listo')
print(f'   Precio:  {df_precio.shape}  y=[{df_precio["y"].min():.1f}, {df_precio["y"].max():.1f}] €/MWh')
print(f'   Demanda: {df_demanda.shape} y=[{df_demanda["y"].min():.0f}, {df_demanda["y"].max():.0f}] MW')
print(f'   n_val={n_val} | n_test={n_test}')

---
## 2. Función para generar covariables futuras

Las covariables futuras son **deterministas**: hora, día de la semana, festivos
y encoding cíclico se pueden calcular para cualquier fecha futura sin descargar nada.
Esta función reemplaza al merge que fallaba porque el dataset no incluye fechas futuras.

In [ ]:
def generar_future_covs(future_df: pd.DataFrame, future_covs: list) -> pd.DataFrame:
    """
    Genera las covariables futuras para los timestamps en future_df.
    Calcula directamente hora, festivos y encodings cíclicos.
    No depende de datos externos — son completamente deterministas.
    """
    df = future_df[['unique_id', 'ds']].copy()

    # Features temporales básicas
    df['hora']        = df['ds'].dt.hour
    df['dia_semana']  = df['ds'].dt.dayofweek
    df['mes']         = df['ds'].dt.month
    df['anio']        = df['ds'].dt.year
    df['semana_anio'] = df['ds'].dt.isocalendar().week.astype(int)

    # Encoding cíclico
    df['hora_sin']   = np.sin(2 * np.pi * df['hora']        / 24)
    df['hora_cos']   = np.cos(2 * np.pi * df['hora']        / 24)
    df['dia_sin']    = np.sin(2 * np.pi * df['dia_semana']  / 7)
    df['dia_cos']    = np.cos(2 * np.pi * df['dia_semana']  / 7)
    df['mes_sin']    = np.sin(2 * np.pi * df['mes']         / 12)
    df['mes_cos']    = np.cos(2 * np.pi * df['mes']         / 12)
    df['semana_sin'] = np.sin(2 * np.pi * df['semana_anio'] / 52)
    df['semana_cos'] = np.cos(2 * np.pi * df['semana_anio'] / 52)

    # Festivos (Comunidad de Madrid)
    años = df['ds'].dt.year.unique()
    festivos_es = {}
    for año in años:
        festivos_es.update(holidays.Spain(years=int(año), subdiv='MD'))
    fechas_festivo = set(festivos_es.keys())

    df['fecha']    = df['ds'].dt.date
    df['es_festivo'] = df['fecha'].isin(fechas_festivo).astype(float)

    df['dias_hasta_festivo'] = df['fecha'].apply(
        lambda f: min((d - f).days for d in fechas_festivo if d >= f)
        if any(d >= f for d in fechas_festivo) else 365
    )
    df['es_vispera_festivo'] = (df['dias_hasta_festivo'] == 1).astype(float)
    df['es_post_festivo']    = (
        (df['es_festivo'].shift(1, fill_value=0) == 1) &
        (df['es_festivo'] == 0)
    ).astype(float)
    df['es_hora_solar'] = df['hora'].between(10, 16).astype(float)
    df.drop(columns=['fecha'], inplace=True)

    # Añadir solo las columnas que el modelo necesita
    cols_faltantes = [c for c in future_covs if c not in df.columns]
    if cols_faltantes:
        print(f'  ⚠️  Covariables no generadas (se rellenan con 0): {cols_faltantes}')
        for c in cols_faltantes:
            df[c] = 0.0

    # Verificar nulos
    nulos = df[future_covs].isnull().sum().sum()
    if nulos > 0:
        print(f'  ⚠️  Nulos encontrados: {nulos} — rellenando con 0')
        df[future_covs] = df[future_covs].fillna(0)

    return df

print('✅ Función generar_future_covs definida')

---
## 3. Funciones de evaluación

In [ ]:
def mape(y_true, y_pred):
    mask = np.abs(y_true) > 1.0
    if mask.sum() == 0: return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def evaluar_nf(df_pred, df_real, col_pred, variable, n_eval=N_EVAL):
    df_m = df_pred.merge(
        df_real[['unique_id','ds','y']], on=['unique_id','ds'], how='inner'
    ).dropna(subset=[col_pred, 'y']).sort_values('ds')
    ds_test = df_m['ds'].unique()[-n_test:]
    ds_eval = ds_test[:n_eval]
    df_eval = df_m[df_m['ds'].isin(ds_eval)]
    y_true  = df_eval['y'].values
    y_pred  = df_eval[col_pred].values
    mae_g   = mean_absolute_error(y_true, y_pred)
    rmse_g  = np.sqrt(mean_squared_error(y_true, y_pred))
    mape_g  = mape(y_true, y_pred)
    unidad  = '€/MWh' if variable == 'precio' else 'MW'
    nombre  = col_pred.split('/')[0] if '/' in col_pred else col_pred
    print(f'\n{"="*52}\n{nombre} — {variable}\n{"="*52}')
    print(f'  MAE:  {mae_g:>10.2f} {unidad}')
    print(f'  RMSE: {rmse_g:>10.2f} {unidad}')
    print(f'  MAPE: {mape_g:>10.2f}%')
    return {'nombre': nombre, 'variable': variable,
            'mae': mae_g, 'rmse': rmse_g, 'mape': mape_g,
            'mae_h': [], 'rmse_h': []}

def plot_pred_nf(df_pred, df_real, col_pred, variable, n_dias=7):
    df_m = df_pred.merge(
        df_real[['ds','y']], on='ds', how='inner'
    ).dropna(subset=[col_pred]).sort_values('ds')
    df_plot = df_m.tail(n_test).head(n_dias * 24)
    fig, ax = plt.subplots(figsize=(16, 4))
    ax.plot(range(len(df_plot)), df_plot['y'].values,
            color=COLORS[0], lw=1.2, label='Real')
    ax.plot(range(len(df_plot)), df_plot[col_pred].values,
            color=COLORS[3], lw=1.2, ls='--', label='Predicción')
    nombre = col_pred.split('/')[0] if '/' in col_pred else col_pred
    ax.set_title(f'{nombre} — {variable} — primeros {n_dias} días del test', fontweight='bold')
    ax.set_xlabel('Horas')
    ax.set_ylabel('€/MWh' if variable == 'precio' else 'MW')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'data/processed/nb05_{nombre.lower().replace(" ","_")}_{variable}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

print('✅ Funciones de evaluación definidas')

---
## 4. Modelo 1 — N-HiTS

Neural Hierarchical Interpolation for Time Series (Challu et al., NeurIPS 2022).

Tres pilas que operan a diferentes resoluciones temporales:
- **Stack 1** (pooling=168h): tendencia semanal
- **Stack 2** (pooling=24h): estacionalidad diaria
- **Stack 3** (pooling=1h): variaciones horarias

La predicción final es la suma de las tres contribuciones.

In [ ]:
nhits_p = NHITS(
    h                         = HORIZONTE,
    input_size                = 168,
    futr_exog_list            = FUTURE_COVS,
    hist_exog_list            = HIST_COVS,
    loss                      = HuberLoss(),
    stack_types               = ['identity', 'identity', 'identity'],
    n_blocks                  = [3, 3, 3],
    mlp_units                 = [[512, 512], [512, 512], [512, 512]],
    n_pool_kernel_size        = [168, 24, 1],
    n_freq_downsample         = [24, 4, 1],
    learning_rate             = 1e-3,
    max_steps                 = 1000,
    batch_size                = 32,
    val_check_steps           = 100,
    early_stop_patience_steps = 5,
    scaler_type               = 'robust',
    random_seed               = 42,
)

nf_nhits_p = NeuralForecast(models=[nhits_p], freq='h')
print('Entrenando N-HiTS precio...')
nf_nhits_p.fit(df=df_precio, val_size=n_val)
print('✅ N-HiTS precio entrenado')

In [ ]:
# Generar futr_df con covariables calculadas directamente
future_nhits_p = generar_future_covs(
    nf_nhits_p.make_future_dataframe(df=df_precio),
    FUTURE_COVS
)
print(f'future_df shape: {future_nhits_p.shape} | Nulos: {future_nhits_p[FUTURE_COVS].isnull().sum().sum()}')

pred_nhits_p = nf_nhits_p.predict(futr_df=future_nhits_p).reset_index()
col_nhits_p  = [c for c in pred_nhits_p.columns if c not in ['unique_id','ds']][0]
print(f'Columna predicción: {col_nhits_p}')

res_nhits_precio = evaluar_nf(pred_nhits_p, df_precio, col_nhits_p, 'precio')
plot_pred_nf(pred_nhits_p, df_precio, col_nhits_p, 'precio')

In [ ]:
nhits_d = NHITS(
    h                         = HORIZONTE,
    input_size                = 168,
    futr_exog_list            = FUTURE_COVS,
    hist_exog_list            = HIST_COVS,
    loss                      = MAE(),
    stack_types               = ['identity', 'identity', 'identity'],
    n_blocks                  = [3, 3, 3],
    mlp_units                 = [[512, 512], [512, 512], [512, 512]],
    n_pool_kernel_size        = [168, 24, 1],
    n_freq_downsample         = [24, 4, 1],
    learning_rate             = 1e-3,
    max_steps                 = 1000,
    batch_size                = 32,
    val_check_steps           = 100,
    early_stop_patience_steps = 5,
    scaler_type               = 'standard',
    random_seed               = 42,
)

nf_nhits_d = NeuralForecast(models=[nhits_d], freq='h')
print('Entrenando N-HiTS demanda...')
nf_nhits_d.fit(df=df_demanda, val_size=n_val)
print('✅ N-HiTS demanda entrenado')

In [ ]:
future_nhits_d = generar_future_covs(
    nf_nhits_d.make_future_dataframe(df=df_demanda),
    FUTURE_COVS
)
print(f'future_df shape: {future_nhits_d.shape} | Nulos: {future_nhits_d[FUTURE_COVS].isnull().sum().sum()}')

pred_nhits_d = nf_nhits_d.predict(futr_df=future_nhits_d).reset_index()
col_nhits_d  = [c for c in pred_nhits_d.columns if c not in ['unique_id','ds']][0]

res_nhits_demanda = evaluar_nf(pred_nhits_d, df_demanda, col_nhits_d, 'demanda')
plot_pred_nf(pred_nhits_d, df_demanda, col_nhits_d, 'demanda')

In [ ]:
import os
os.makedirs('data/models/nf', exist_ok=True)
nf_nhits_p.save('data/models/nf/nhits_precio',  overwrite=True)
nf_nhits_d.save('data/models/nf/nhits_demanda', overwrite=True)
print('✅ N-HiTS guardado en data/models/nf/')

---
## 5. Modelo 2 — TFT (Temporal Fusion Transformer)

Lim et al. (Google, NeurIPS 2020). Arquitectura híbrida que combina:
- **Variable Selection Networks**: aprende qué features son más relevantes
- **LSTM encoder/decoder**: procesa la secuencia temporal
- **Multi-head attention**: captura dependencias de largo alcance
- **Gating mechanisms**: filtra información irrelevante

Ventaja clave sobre LSTM: las covariables futuras conocidas (hora, festivos)
se procesan de forma **separada** a las pasadas observadas (temperatura, lags),
lo que mejora especialmente la predicción a horizontes largos (h+24, h+48).

In [ ]:
tft_p = TFT(
    h                         = HORIZONTE,
    input_size                = 168,
    futr_exog_list            = FUTURE_COVS,
    hist_exog_list            = HIST_COVS,
    loss                      = HuberLoss(),
    hidden_size               = 128,
    n_head                    = 4,
    attn_dropout              = 0.1,
    dropout                   = 0.1,
    learning_rate             = 1e-3,
    max_steps                 = 1000,
    batch_size                = 32,
    val_check_steps           = 100,
    early_stop_patience_steps = 5,
    scaler_type               = 'robust',
    random_seed               = 42,
)

nf_tft_p = NeuralForecast(models=[tft_p], freq='h')
print('Entrenando TFT precio...')
nf_tft_p.fit(df=df_precio, val_size=n_val)
print('✅ TFT precio entrenado')

In [ ]:
future_tft_p = generar_future_covs(
    nf_tft_p.make_future_dataframe(df=df_precio),
    FUTURE_COVS
)
print(f'future_df shape: {future_tft_p.shape} | Nulos: {future_tft_p[FUTURE_COVS].isnull().sum().sum()}')

pred_tft_p  = nf_tft_p.predict(futr_df=future_tft_p).reset_index()
col_tft_p   = [c for c in pred_tft_p.columns if c not in ['unique_id','ds']][0]
print(f'Columna predicción: {col_tft_p}')

res_tft_precio = evaluar_nf(pred_tft_p, df_precio, col_tft_p, 'precio')
plot_pred_nf(pred_tft_p, df_precio, col_tft_p, 'precio')

In [ ]:
tft_d = TFT(
    h                         = HORIZONTE,
    input_size                = 168,
    futr_exog_list            = FUTURE_COVS,
    hist_exog_list            = HIST_COVS,
    loss                      = MAE(),
    hidden_size               = 128,
    n_head                    = 4,
    attn_dropout              = 0.1,
    dropout                   = 0.1,
    learning_rate             = 1e-3,
    max_steps                 = 1000,
    batch_size                = 32,
    val_check_steps           = 100,
    early_stop_patience_steps = 5,
    scaler_type               = 'standard',
    random_seed               = 42,
)

nf_tft_d = NeuralForecast(models=[tft_d], freq='h')
print('Entrenando TFT demanda...')
nf_tft_d.fit(df=df_demanda, val_size=n_val)
print('✅ TFT demanda entrenado')

In [ ]:
future_tft_d = generar_future_covs(
    nf_tft_d.make_future_dataframe(df=df_demanda),
    FUTURE_COVS
)
print(f'future_df shape: {future_tft_d.shape} | Nulos: {future_tft_d[FUTURE_COVS].isnull().sum().sum()}')

pred_tft_d  = nf_tft_d.predict(futr_df=future_tft_d).reset_index()
col_tft_d   = [c for c in pred_tft_d.columns if c not in ['unique_id','ds']][0]

res_tft_demanda = evaluar_nf(pred_tft_d, df_demanda, col_tft_d, 'demanda')
plot_pred_nf(pred_tft_d, df_demanda, col_tft_d, 'demanda')

In [ ]:
nf_tft_p.save('data/models/nf/tft_precio',  overwrite=True)
nf_tft_d.save('data/models/nf/tft_demanda', overwrite=True)
print('✅ TFT guardado en data/models/nf/')

---
## 6. Comparativa final — todos los modelos

In [ ]:
with open('data/models/resultados_v2.json') as f:
    res_nb04 = json.load(f)

def cargar_res(var, modelo, nombre):
    r = res_nb04[var][modelo]
    return {'nombre': nombre, 'variable': var,
            'mae': r['mae'], 'rmse': r['rmse'], 'mape': r['mape'],
            'mae_h': r['mae_h'], 'rmse_h': r['rmse_h']}

res_sarima_p = cargar_res('precio',  'sarima',  'SARIMA')
res_xgb_p    = cargar_res('precio',  'xgboost', 'XGBoost')
res_lstm_p   = cargar_res('precio',  'lstm',    'LSTM')
res_sarima_d = cargar_res('demanda', 'sarima',  'SARIMA')
res_xgb_d    = cargar_res('demanda', 'xgboost', 'XGBoost')
res_lstm_d   = cargar_res('demanda', 'lstm',    'LSTM')

res_nhits_precio['nombre']  = 'N-HiTS'
res_nhits_demanda['nombre'] = 'N-HiTS'
res_tft_precio['nombre']    = 'TFT'
res_tft_demanda['nombre']   = 'TFT'

print('✅ Resultados notebook 04 cargados')

In [ ]:
print('='*65)
print('COMPARATIVA FINAL — TODOS LOS MODELOS')
print('='*65)

for titulo, resultados in [
    ('PRECIO (€/MWh)', [res_sarima_p, res_xgb_p, res_lstm_p, res_nhits_precio, res_tft_precio]),
    ('DEMANDA (MW)',   [res_sarima_d, res_xgb_d, res_lstm_d, res_nhits_demanda, res_tft_demanda])
]:
    print(f'\n{titulo}')
    print(f'{"Modelo":<14}{"MAE":>10}{"RMSE":>10}{"MAPE%":>10}')
    print('-'*46)
    mejor_mae = min(r['mae'] for r in resultados)
    for r in resultados:
        marca = ' ★' if r['mae'] == mejor_mae else ''
        print(f'{r["nombre"]+marca:<14}{r["mae"]:>10.2f}{r["rmse"]:>10.2f}{r["mape"]:>10.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Comparativa MAE — SARIMA → XGBoost → LSTM → N-HiTS → TFT', fontweight='bold')

nombres = ['SARIMA', 'XGBoost', 'LSTM', 'N-HiTS', 'TFT']

for ax, (titulo, resultados) in zip(axes, [
    ('Precio (€/MWh)', [res_sarima_p, res_xgb_p, res_lstm_p, res_nhits_precio, res_tft_precio]),
    ('Demanda (MW)',   [res_sarima_d, res_xgb_d, res_lstm_d, res_nhits_demanda, res_tft_demanda])
]):
    maes = [r['mae'] for r in resultados]
    bars = ax.bar(nombres, maes, color=COLORS[:5], alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, maes):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
                f'{val:.2f}', ha='center', fontsize=9, fontweight='bold')
    idx_mejor = int(np.argmin(maes))
    bars[idx_mejor].set_edgecolor('black')
    bars[idx_mejor].set_linewidth(2.5)
    ax.set_title(titulo); ax.set_ylabel('MAE')
    ax.set_xticklabels(nombres, rotation=15)

plt.tight_layout()
plt.savefig('data/processed/nb05_comparativa_final.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
todos_p = [res_sarima_p, res_xgb_p, res_lstm_p, res_nhits_precio, res_tft_precio]
todos_d = [res_sarima_d, res_xgb_d, res_lstm_d, res_nhits_demanda, res_tft_demanda]

mejor_p  = min(todos_p, key=lambda x: x['mae'])
mejor_d  = min(todos_d, key=lambda x: x['mae'])
mejora_p = (res_sarima_p['mae'] - mejor_p['mae']) / res_sarima_p['mae'] * 100
mejora_d = (res_sarima_d['mae'] - mejor_d['mae']) / res_sarima_d['mae'] * 100

print('='*65)
print('CONCLUSIONES FINALES — TFM')
print('='*65)
print(f'''
PRECIO:
  Mejor modelo:        {mejor_p["nombre"]} (MAE = {mejor_p["mae"]:.2f} €/MWh)
  Mejora sobre SARIMA: {mejora_p:.1f}%

DEMANDA:
  Mejor modelo:        {mejor_d["nombre"]} (MAE = {mejor_d["mae"]:.2f} MW)
  Mejora sobre SARIMA: {mejora_d:.1f}%

PROGRESIÓN ACADÉMICA:
  SARIMA   → baseline estadístico clásico
  XGBoost  → ML tabular con SHAP (interpretabilidad)
  LSTM     → DL recurrente (168h lookback, Huber loss)
  N-HiTS   → DL jerárquico multihorizonte (NeurIPS 2022)
  TFT      → Transformer con atención temporal (Google, NeurIPS 2020)
''')

resultados_finales = {}
for var, res_list in [('precio', todos_p), ('demanda', todos_d)]:
    resultados_finales[var] = {}
    for r in res_list:
        resultados_finales[var][r['nombre'].lower().replace('-','_').replace(' ','_')] = {
            'mae': float(r['mae']), 'rmse': float(r['rmse']), 'mape': float(r['mape'])
        }

with open('data/models/resultados_finales.json', 'w') as f:
    json.dump(resultados_finales, f, indent=2)

print('✅ Notebook 05 completado.')
print('   Siguiente paso: app Streamlit')